## Simple Steam prices prediction
* [Dataset](https://www.kaggle.com/datasets/artyomkruglov/gaming-profiles-2025-steam-playstation-xbox):  archive/steam/\[ games.csv, prices.csv \] -> ./dataset/\[ games.csv, prices.csv \]

In [29]:
from ast import literal_eval

from tqdm.notebook import tqdm

import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

### Settings

In [2]:
available_currencies = ["usd", "eur", "gbp", "jpy", "rub"]
picked_cur = available_currencies[0]
max_price = 100

picked_cur, max_price

('usd', 100)

### Dataset work

In [3]:
games = pd.read_csv("./dataset/games.csv")
prices = pd.read_csv("./dataset/prices.csv")

In [4]:
print(games.size)
games.head(10)

687736


,gameid,title,developers,publishers,genres,supported_languages,release_date
0,3281560,Horror Game To Play With Friends! Playtest,NaN,NaN,NaN,NaN,2024-10-21
1,3280930,Eternals' Path Playtest,NaN,NaN,NaN,NaN,2024-10-17
2,3280770,ANGST: A TALE OF SURVIVAL - Singleplayer Playtest,NaN,NaN,NaN,NaN,2024-10-13
3,3279790,Montabi Playtest,NaN,NaN,NaN,NaN,2024-10-13
4,3278320,파이팅걸 유리 Playtest,NaN,NaN,NaN,NaN,2024-10-12
5,3278740,NEURO,['Revolt Games'],['Strategy First'],['Action'],"['English', 'Russian']",2024-10-11
6,3277430,Objective: F.E.A.S.T. Playtest,NaN,NaN,NaN,NaN,2024-10-12
7,3276500,Fortune Avenue Playtest,NaN,NaN,NaN,NaN,2024-10-09
8,3274370,Beyond the Ordinary Playtest,NaN,NaN,NaN,NaN,2024-10-11
9,3274670,Quantum Joe Playtest,NaN,NaN,NaN,NaN,2024-10-11


In [5]:
print(prices.size)
prices.head(10)

30899911


,gameid,usd,eur,gbp,jpy,rub,date_acquired
0,3281560,NaN,NaN,NaN,NaN,NaN,2024-11-28
1,3280930,NaN,NaN,NaN,NaN,NaN,2024-11-28
2,3280770,NaN,NaN,NaN,NaN,NaN,2024-11-28
3,3279790,NaN,NaN,NaN,NaN,NaN,2024-11-28
4,3278320,NaN,NaN,NaN,NaN,NaN,2024-11-28
5,3278740,5.99,5.85,5.1,720.0,228.0,2024-11-28
6,3277430,NaN,NaN,NaN,NaN,NaN,2024-11-28
7,3276500,NaN,NaN,NaN,NaN,NaN,2024-11-28
8,3274370,NaN,NaN,NaN,NaN,NaN,2024-11-28
9,3274670,NaN,NaN,NaN,NaN,NaN,2024-11-28


In [6]:
merged = pd.merge(games, prices, on=['gameid'], how='inner')

In [7]:
merged = merged.dropna(subset=['release_date', picked_cur])

In [8]:
merged.title.str.lower().str.contains("playtest").any()

False

In [9]:
merged = merged[merged[picked_cur] <= max_price]

In [10]:
merged = merged.sort_values('release_date')
# merged = merged.sort_values('date_acquired')

In [11]:
merged['release_date'] = pd.to_datetime(merged['release_date'], errors='coerce')
merged['date_acquired'] = pd.to_datetime(merged['date_acquired'], errors='coerce')
merged = merged.reset_index()

In [12]:
print(merged.size)
merged.head()

48905668


,index,gameid,title,developers,publishers,genres,supported_languages,release_date,usd,eur,gbp,jpy,rub,date_acquired
0,3838364,282010,Carmageddon Max Pack,['Stainless Games Ltd'],['THQ Nordic'],"['Action', 'Indie', 'Racing']",['English'],1997-06-30,9.99,NaN,5.99,980.0,249.0,2025-02-24
1,3838338,282010,Carmageddon Max Pack,['Stainless Games Ltd'],['THQ Nordic'],"['Action', 'Indie', 'Racing']",['English'],1997-06-30,9.99,NaN,5.99,980.0,249.0,2025-01-03
2,3838337,282010,Carmageddon Max Pack,['Stainless Games Ltd'],['THQ Nordic'],"['Action', 'Indie', 'Racing']",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2025-01-01
3,3838336,282010,Carmageddon Max Pack,['Stainless Games Ltd'],['THQ Nordic'],"['Action', 'Indie', 'Racing']",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2024-12-30
4,3838335,282010,Carmageddon Max Pack,['Stainless Games Ltd'],['THQ Nordic'],"['Action', 'Indie', 'Racing']",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2024-12-28


In [13]:
merged['genres'] = merged['genres'].fillna("[]").apply(literal_eval)
merged['developers'] = merged['developers'].fillna("[]").apply(literal_eval)
merged['publishers'] = merged['publishers'].fillna("[]").apply(literal_eval)

In [14]:
# max_time = merged[['release_date', 'date_acquired']].max().max()

merged['days_since_release'] = (merged['date_acquired'] - merged['release_date']).dt.days

In [15]:
merged.head()

,index,gameid,title,developers,publishers,genres,supported_languages,release_date,usd,eur,gbp,jpy,rub,date_acquired,days_since_release
0,3838364,282010,Carmageddon Max Pack,[Stainless Games Ltd],[THQ Nordic],"[Action, Indie, Racing]",['English'],1997-06-30,9.99,NaN,5.99,980.0,249.0,2025-02-24,10101
1,3838338,282010,Carmageddon Max Pack,[Stainless Games Ltd],[THQ Nordic],"[Action, Indie, Racing]",['English'],1997-06-30,9.99,NaN,5.99,980.0,249.0,2025-01-03,10049
2,3838337,282010,Carmageddon Max Pack,[Stainless Games Ltd],[THQ Nordic],"[Action, Indie, Racing]",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2025-01-01,10047
3,3838336,282010,Carmageddon Max Pack,[Stainless Games Ltd],[THQ Nordic],"[Action, Indie, Racing]",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2024-12-30,10045
4,3838335,282010,Carmageddon Max Pack,[Stainless Games Ltd],[THQ Nordic],"[Action, Indie, Racing]",['English'],1997-06-30,2.49,NaN,1.49,245.0,62.0,2024-12-28,10043


In [16]:
def transform_to_id(df, name) -> tuple[pd.Series, dict, list]:
    s = df[name]

    unique = s.explode().unique()
    vocab = { n: i for i, n in enumerate(unique) }

    s = s.map(lambda x: [vocab[y] for y in x])

    return s, vocab, unique

In [17]:
genres_id, genres_vocab, _ = transform_to_id(merged, 'genres')

In [18]:
devs_id, devs_vocab, _ = transform_to_id(merged, 'developers')

In [19]:
pubs_id, pubs_vocab, _ = transform_to_id(merged, 'publishers')

In [20]:
merged['genres'] = genres_id
merged['developers'] = devs_id
merged['publishers'] = pubs_id

In [21]:
merged['year'] = merged['date_acquired'].dt.year
merged['month'] = merged['date_acquired'].dt.month
merged['dayofweek'] = merged['date_acquired'].dt.dayofweek

In [22]:
merged['release_date'] = merged['release_date'].dt.year

In [23]:
merged['month_sin'] = np.sin(2 * np.pi * merged['month'] / 12)
merged['month_cos'] = np.cos(2 * np.pi * merged['month'] / 12)
merged['dow_sin'] = np.sin(2 * np.pi * merged['dayofweek'] / 7)
merged['dow_cos'] = np.cos(2 * np.pi * merged['dayofweek'] / 7)

In [24]:
# merged = merged.drop(["index", "title", "supported_languages", "release_date", *set(available_currencies).difference(picked_cur), "date_acquired"], axis=1)

In [25]:
merged

,index,gameid,title,developers,publishers,genres,supported_languages,release_date,usd,eur,...,rub,date_acquired,days_since_release,year,month,dayofweek,month_sin,month_cos,dow_sin,dow_cos
0,3838364,282010,Carmageddon Max Pack,[0],[0],"[0, 1, 2]",['English'],1997,9.99,NaN,...,249.0,2025-02-24,10101,2025,2,0,8.660254e-01,0.500000,0.000000,1.000000
1,3838338,282010,Carmageddon Max Pack,[0],[0],"[0, 1, 2]",['English'],1997,9.99,NaN,...,249.0,2025-01-03,10049,2025,1,4,5.000000e-01,0.866025,-0.433884,-0.900969
2,3838337,282010,Carmageddon Max Pack,[0],[0],"[0, 1, 2]",['English'],1997,2.49,NaN,...,62.0,2025-01-01,10047,2025,1,2,5.000000e-01,0.866025,0.974928,-0.222521
3,3838336,282010,Carmageddon Max Pack,[0],[0],"[0, 1, 2]",['English'],1997,2.49,NaN,...,62.0,2024-12-30,10045,2024,12,0,-2.449294e-16,1.000000,0.000000,1.000000
4,3838335,282010,Carmageddon Max Pack,[0],[0],"[0, 1, 2]",['English'],1997,2.49,NaN,...,62.0,2024-12-28,10043,2024,12,5,-2.449294e-16,1.000000,-0.974928,-0.222521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493257,4068106,3101380,Base Blitz,[49782],[41965],"[0, 6, 1, 4]",['English'],2025,6.99,6.89,...,280.0,2025-01-31,21,2025,1,4,5.000000e-01,0.866025,-0.433884,-0.900969
3493258,4068107,3101380,Base Blitz,[49782],[41965],"[0, 6, 1, 4]",['English'],2025,6.99,6.89,...,280.0,2025-02-02,23,2025,2,6,8.660254e-01,0.500000,-0.781831,0.623490
3493259,4068108,3101380,Base Blitz,[49782],[41965],"[0, 6, 1, 4]",['English'],2025,6.99,6.89,...,280.0,2025-02-04,25,2025,2,1,8.660254e-01,0.500000,0.781831,0.623490
3493260,4070247,3209910,Slender Reborn,[26588],[17331],"[0, 5, 1, 7]",['English'],2025,3.99,3.99,...,165.0,2025-02-04,25,2025,2,1,8.660254e-01,0.500000,0.781831,0.623490


In [26]:
SEQUENCE_LENGTH = 4
FEATURE_COLS = ['days_since_release', 'month_sin','month_cos','dow_sin','dow_cos']
L =  len(FEATURE_COLS)

In [ ]:
def create_sequences_for_game(game_df, sequence_length=SEQUENCE_LENGTH, feature_cols=FEATURE_COLS, target_col=picked_cur):
    """
    Для одной игры (отсортированной по дате) создаёт окна.
    Возвращает:
        X_seq: список окон (L x num_features)
        X_static: список статических признаков (одинаков для всех окон игры)
        y: целевые значения (цена на следующий шаг)
        также списки IDs для категорий
    """
    if len(game_df) < sequence_length + 1:
        return [[]]*6
    
    dev_ids = game_df.iloc[0]['developers']
    pub_ids = game_df.iloc[0]['publishers']
    gen_ids = game_df.iloc[0]['genres']
    release_year = game_df.iloc[0]['release_date']
    
    X_seq = []
    y = []
    # список окон будем собирать
    for i in range(len(game_df) - sequence_length):
        # берём L записей как вход
        window = game_df.iloc[i:i+sequence_length][feature_cols].values  # shape (SL, num_features)
        target = game_df.iloc[i+sequence_length][target_col]
        X_seq.append(window)
        y.append(target)
    
    # статические данные повторяются для каждого окна
    static_dev = [dev_ids] * len(X_seq)
    static_pub = [pub_ids] * len(X_seq)
    static_gen = [gen_ids] * len(X_seq)
    static_release_year = [release_year] * len(X_seq)
    
    return X_seq, y, static_dev, static_pub, static_gen, static_release_year

In [ ]:
all_X_seq = []
all_y = []
all_dev = []
all_pub = []
all_gen = []
all_release_year = []

for gameid, group in tqdm(merged.sample(frac=0.1, random_state=42).groupby('gameid')):
    # группировка уже отсортирована
    res = create_sequences_for_game(group, SEQUENCE_LENGTH)
    X_seq, y, dev, pub, gen, rel_year = res
    if len(X_seq) > 0:
        all_X_seq.extend(X_seq)
        all_y.extend(y)
        all_dev.extend(dev)
        all_pub.extend(pub)
        all_gen.extend(gen)
        all_release_year.extend(rel_year)

X_seq = np.array(all_X_seq)
y = np.array(all_y)

  0%|          | 0/77368 [00:00<?, ?it/s]

In [44]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from keras.layers import Input, LSTM, Concatenate, Dense, Embedding, GlobalAveragePooling1D

In [ ]:
num_features = X_seq.shape[2]
# Создаём один scaler на все числовые признаки (обучаем на всём X_seq)
scaler = MinMaxScaler(feature_range=(0, 1))
# Нужно привести X_seq к 2D для fit (n_samples*SL, num_features)
original_shape = X_seq.shape
X_seq_flat = X_seq.reshape(-1, num_features)
scaler.fit(X_seq_flat)
X_seq_scaled = scaler.transform(X_seq_flat).reshape(original_shape)

# ============================================
# 6. Подготовка категориальных последовательностей (паддинг)
# ============================================
# Для каждого списка IDs используем паддинг до максимальной длины в датасете
max_dev_len = max(len(lst) for lst in all_dev)
max_pub_len = max(len(lst) for lst in all_pub)
max_gen_len = max(len(lst) for lst in all_gen)

def pad_ids(ids_list, max_len):
    return np.array([lst + [0]*(max_len - len(lst)) for lst in ids_list])

dev_padded = pad_ids(all_dev, max_dev_len)
pub_padded = pad_ids(all_pub, max_pub_len)
gen_padded = pad_ids(all_gen, max_gen_len)

In [52]:
def embedding_layer(vocab: dict, output_dim: int = 8, name: str=None) -> tuple:
    inp = Input(shape=(None,), name=f"{name}_inp")
    emb = Embedding(input_dim=len(vocab), output_dim=output_dim, name=f"{name}_emb")(inp)
    pool = GlobalAveragePooling1D(name=f"{name}_pool")(emb)
    return inp, pool

In [53]:
seq_input = Input(shape=(SEQUENCE_LENGTH, L))
lstm_out = LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.2)(seq_input)